# Tratamento das Bases semanais do RH

**Inputs**\
Planilhas semanais enviadas pelo time de RH com dados dos colaboradores desligados

**Outputs**\
Planilha unica com os desligados dos ultimos 24 meses + ativos


In [0]:
%run "/Workspace/Jurídico/funcao_tratamento_fechamento/common_functions"

  Obtaining dependency information for openpyxl from https://files.pythonhosted.org/packages/c0/da/977ded879c29cbd04de313843e76868e6e13408a94ed6b987245dc7c8506/openpyxl-3.1.5-py2.py3-none-any.whl.metadata
  Obtaining dependency information for et-xmlfile from https://files.pythonhosted.org/packages/c1/8b/5fe2cc11fee489817272089c4203e679c63b570a5aaeb18d852ae3cbba6a/et_xmlfile-2.0.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/250.9 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━ 225.3/250.9 kB 6.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 5.5 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


  Obtaining dependency information for xlsxwriter from https://files.pythonhosted.org/packages/3a/0c/3662f4a66880196a590b202f0db82d919dd2f89e99a27fadef91c4a33d41/xlsxwriter-3.2.9-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/175.3 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 174.1/175.3 kB 5.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


  Obtaining dependency information for unidecode from https://files.pythonhosted.org/packages/8f/b7/559f59d57d18b44c6d1250d2eeaa676e028b9c527431f5d0736478a73ba1/Unidecode-1.4.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/235.8 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 235.5/235.8 kB 7.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Parametro do nome da tabela da competência atual. Ex: 202404
dbutils.widgets.text("nmmes", "")

# Parametro nome do arquivo enviado pelo RH com os desligados. Ex: 20240423
dbutils.widgets.text("dtdesligados", "")

In [0]:
dtdesligados = dbutils.widgets.get("dtdesligados")
nmmes = dbutils.widgets.get("nmmes")

caminho_desligados = f"/Volumes/databox/juridico_comum/arquivos/trabalhista/bases_rh/base_desligados/RH_BASE_DESLIGADOS_{dtdesligados}.xlsx" 

caminho_rh_previa = f"/Volumes/databox/juridico_comum/arquivos/trabalhista/bases_rh/previa/"

In [0]:
df_desligados = read_excel(caminho_desligados,"'BASE_DESLIGADOS'!A1:AD1048576")

In [0]:
import os
from pyspark.sql import DataFrame

caminho = "/Volumes/databox/juridico_comum/arquivos/trabalhista/bases_rh/previa/"

files = dbutils.fs.ls(caminho)

dataframes = {}

for f in files:
    if "PREVIA_SEMANAL_RH_" in f.name:
        nome_df = os.path.splitext(f.name)[0]  # nome do arquivo sem extensão
        caminho_arquivo = f.path

        df = spark.read \
            .format("com.crealytics.spark.excel") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(caminho_arquivo)

        dataframes[nome_df] = df


df_previa_rh_final: DataFrame = None

for df in dataframes.values():
    if df_previa_rh_final is None:
        df_previa_rh_final = df
    else:
        df_previa_rh_final = df_previa_rh_final.unionByName(df, allowMissingColumns=True)

colunas_validas = ["MATRICULA", "STATUS_Colaborador", "DATA_ADMISSAO","DATA_DESLIGAMENTO"]  # substitua pelas colunas desejadas
df_previa_rh_final = df_previa_rh_final.dropDuplicates(colunas_validas)

df_previa_rh_final = df_previa_rh_final.toDF(*[col.upper() for col in df_previa_rh_final.columns])

df_previa_rh_final = df_previa_rh_final.drop("EMAIL")

df_previa_rh_final.createOrReplaceTempView('df_previa_rh_final')  

In [0]:
print(df_previa_rh_final.columns)

['COD_EMPRESA', 'MATRICULA', 'NOME', 'CPF', 'ESCOLARIDADE', 'GENERO', 'ETNIA', 'DATA_NASCIMENTO', 'STATUS_COLABORADOR', 'DATA_ADMISSAO', 'TEMPO_EMPRESA', 'DESC_FUNCAO_ATUAL', 'DATA_DESLIGAMENTO', 'TIPO_DESLIGAMENTO', 'MOTIVO_DESLIGAMENTO', 'DESC_TIPO_DEFICIENCIA', 'AGRUP_FUNCAO', 'MATRICULA_GESTOR', 'NOME_GESTOR', 'COD_CENTRO_RESULTADO', 'DESC_CENTRO_RESULTADO', 'COD_CENTRO_CUSTO_FORM', 'ESTRUTURA', 'NIVEL1', 'NIVEL2', 'NIVEL3', 'NIVEL4', 'NIVEL5', 'COD_ESTABELECIMENTO', 'COD_CENTRO_CUSTO', 'ANO_MES', 'TEMPO_CASA', 'DESC_MOTIVO_AFASTAMENTO', 'AGRUP_CARGOS_FUNCOES', ',MOJ']


#### Base de desligados historica abaixo

In [0]:
print(df_desligados.columns)

['CodEmpresa', 'MATRICULA', 'NOME', 'CPF', 'ESCOLARIDADE', 'GENERO', 'ETNIA', 'Nascimento', 'Status', 'Admissao', 'TempoEmpresa', 'Funcao', 'Desligamento', 'TipoDesligamento', 'MOTIVO_DESLIGAMENTO', 'TipoDeficiencia', 'AgrupamentoCargo', 'MatriculaGestor', 'NomeGestor', 'CodCResultado', 'DESC_CENTRO_RESULTADO', 'CodCCusto', 'ESTRUTURA', 'DescricaoN1', 'DescricaoN2', 'DescricaoN3', 'DescricaoN4', 'DescricaoN5', 'Estabelecimento']


In [0]:
from pyspark.sql.functions import *

try:
    # Um dicionário mapeando nome da coluna para o novo tipo
    novos_tipos = {
        "CodEmpresa": "integer",
        "MATRICULA": "bigint",
        "CPF": "bigint",
        "Nascimento": "date",
        "Admissao": "date",
        "Desligamento": "date"
    }

    # Aplicando os casts
    for coluna, tipo in novos_tipos.items():
        df_desligados = df_desligados.withColumn(coluna, col(coluna).cast(tipo))
    
    # Formatando as datas para remover a hora
    colunas_data = ["Nascimento", "Admissao", "Desligamento"]
    for coluna in colunas_data:
        df_desligados = df_desligados.withColumn(coluna, date_format(col(coluna), "yyyy-MM-dd"))



    df_desligados = df_desligados.toDF(*[col.upper() for col in df_desligados.columns])

    renomear_colunas = {
        "CODEMPRESA": "COD_EMPRESA",
        "NASCIMENTO": "DATA_NASCIMENTO",
        "STATUS": "STATUS_COLABORADOR",
        "ADMISSAO": "DATA_ADMISSAO",
        "TEMPOEMPRESA": "TEMPO_EMPRESA",
        "FUNCAO": "DESC_FUNCAO_ATUAL",
        "DESLIGAMENTO": "DATA_DESLIGAMENTO",
        "TIPODESLIGAMENTO": "TIPO_DESLIGAMENTO",
        "TIPODEFICIENCIA": "DESC_TIPO_DEFICIENCIA",
        "AGRUPAMENTOCARGO": "AGRUP_FUNCAO",
        "MATRICULAGESTOR": "MATRICULA_GESTOR",
        "NOMEGESTOR": "NOME_GESTOR",
        "CODCRESULTADO": "COD_CENTRO_RESULTADO",
        "CODCCUSTO": "COD_CENTRO_CUSTO_FORM",
        "DESCRICAON1":"NIVEL1",
        "DESCRICAON2":"NIVEL2",
        "DESCRICAON3":"NIVEL3",
        "DESCRICAON4":"NIVEL4",
        "DESCRICAON5":"NIVEL5",
        "ESTABELECIMENTO":"COD_ESTABELECIMENTO",
    }

    # Aplicar os renomes
    for antigo, novo in renomear_colunas.items():
        df_desligados = df_desligados.withColumnRenamed(antigo, novo)
    
    # NOVA PARTE: Formatando as datas para remover a hora (depois de renomear)
    colunas_data_formatar = ["DATA_NASCIMENTO", "DATA_ADMISSAO", "DATA_DESLIGAMENTO"]
    for coluna in colunas_data_formatar:
        if coluna in df_desligados.columns:
            df_desligados = df_desligados.withColumn(coluna, date_format(col(coluna), "yyyy-MM-dd"))



    df_desligados.createOrReplaceTempView('df_desligados')

except Exception as e:
    print(e)

In [0]:
# Define the common columns
common_cols = [col.upper() for col in df_desligados.columns if col.upper() in [c.upper() for c in df_previa_rh_final.columns]]

# Select only the common columns from both DataFrames
df_desligados_sel = df_desligados.select(*common_cols)
df_previa_rh_final_sel = df_previa_rh_final.select(*common_cols)

# Register temp views
df_desligados_sel.createOrReplaceTempView('df_desligados_sel')
df_previa_rh_final_sel.createOrReplaceTempView('df_previa_rh_final_sel')

# Perform UNION
df_rh_hist_com_previa = spark.sql('''
    SELECT * FROM df_desligados_sel
    UNION
    SELECT * FROM df_previa_rh_final_sel
''')

df_rh_hist_com_previa = df_rh_hist_com_previa.withColumn(
    "STATUS_COLABORADOR", 
    upper(df_rh_hist_com_previa["STATUS_COLABORADOR"])
)

df_rh_hist_com_previa = df_rh_hist_com_previa.replace(
    "DESLIGADO-R", 
    "DESLIGADO", 
    subset=["STATUS_COLABORADOR"]
)

colunas_validas = [
    "MATRICULA", 
    "STATUS_COLABORADOR", 
    "DATA_ADMISSAO",
    "DATA_DESLIGAMENTO"
]
df_rh_hist_com_previa = df_rh_hist_com_previa.dropDuplicates(colunas_validas)

df_rh_hist_com_previa = df_rh_hist_com_previa.withColumn("MES_ADMISSAO", trunc("DATA_ADMISSAO", "MM"))

df_rh_hist_com_previa = df_rh_hist_com_previa.withColumn("MES_DESLIGAMENTO", trunc("DATA_DESLIGAMENTO", "MM"))

df_rh_hist_com_previa.createOrReplaceTempView('df_rh_hist_com_previa')

In [0]:
print(df_previa_rh_final.columns)

['COD_EMPRESA', 'MATRICULA', 'NOME', 'CPF', 'ESCOLARIDADE', 'GENERO', 'ETNIA', 'DATA_NASCIMENTO', 'STATUS_COLABORADOR', 'DATA_ADMISSAO', 'TEMPO_EMPRESA', 'DESC_FUNCAO_ATUAL', 'DATA_DESLIGAMENTO', 'TIPO_DESLIGAMENTO', 'MOTIVO_DESLIGAMENTO', 'DESC_TIPO_DEFICIENCIA', 'AGRUP_FUNCAO', 'MATRICULA_GESTOR', 'NOME_GESTOR', 'COD_CENTRO_RESULTADO', 'DESC_CENTRO_RESULTADO', 'COD_CENTRO_CUSTO_FORM', 'ESTRUTURA', 'NIVEL1', 'NIVEL2', 'NIVEL3', 'NIVEL4', 'NIVEL5', 'COD_ESTABELECIMENTO', 'COD_CENTRO_CUSTO', 'ANO_MES', 'TEMPO_CASA', 'DESC_MOTIVO_AFASTAMENTO', 'AGRUP_CARGOS_FUNCOES', ',MOJ']


In [0]:
print(df_desligados.columns)

['COD_EMPRESA', 'MATRICULA', 'NOME', 'CPF', 'ESCOLARIDADE', 'GENERO', 'ETNIA', 'DATA_NASCIMENTO', 'STATUS_COLABORADOR', 'DATA_ADMISSAO', 'TEMPO_EMPRESA', 'DESC_FUNCAO_ATUAL', 'DATA_DESLIGAMENTO', 'TIPO_DESLIGAMENTO', 'MOTIVO_DESLIGAMENTO', 'DESC_TIPO_DEFICIENCIA', 'AGRUP_FUNCAO', 'MATRICULA_GESTOR', 'NOME_GESTOR', 'COD_CENTRO_RESULTADO', 'DESC_CENTRO_RESULTADO', 'COD_CENTRO_CUSTO_FORM', 'ESTRUTURA', 'NIVEL1', 'NIVEL2', 'NIVEL3', 'NIVEL4', 'NIVEL5', 'COD_ESTABELECIMENTO']


In [0]:
df_rh_hist_com_previa_f = spark.sql('''
    SELECT A.* FROM df_rh_hist_com_previa A LEFT JOIN(
    SELECT MATRICULA, MAX(MES_DESLIGAMENTO) AS ULTIMO_DESLIGAMENTO FROM df_rh_hist_com_previa
    GROUP BY ALL) AS B
    ON A.MATRICULA = B.MATRICULA AND A.MES_DESLIGAMENTO = B.ULTIMO_DESLIGAMENTO
''')

df_rh_hist_com_previa_f.createOrReplaceTempView('df_rh_hist_com_previa_f')

In [0]:
from pyspark.sql.functions import col, regexp_replace, lpad, when

# 1. Carrega a 'view' da célula anterior
df = spark.table('df_rh_hist_com_previa_f')

# 2. Converte o CPF para string (é aqui que a notação científica aparece)
df = df.withColumn("CPF_STR", col("CPF").cast("string"))

# 3. Limpa TUDO que não for número (pontos, traços, e o ".0" ou "E10" da notação científica)
df = df.withColumn("CPF_CLEAN", regexp_replace(col("CPF_STR"), r"[^0-9]", ""))

# 4. Cria a coluna 'CPFN' final, garantindo que ela tenha 11 dígitos, com zeros à esquerda
#    Esta é a lógica de limpeza e padding que faltava
df = df.withColumn("CPFN", lpad(col("CPF_CLEAN"), 11, "0"))

# 5. Salva o resultado em 'desligados_1' e cria a 'view' para a Célula 19 usar
desligados_1 = df
desligados_1.createOrReplaceTempView('desligados_1')

In [0]:
#desligados_1 = spark.sql('''
 #   SELECT *,
  #  CASE WHEN LENGTH(CAST(CPF AS STRING)) == 10 THEN CONCAT("0", CAST(CPF AS STRING))
   #         ELSE CAST(CPF AS STRING) END AS CPFN
    #FROM df_rh_hist_com_previa_f
#''')

#desligados_1.createOrReplaceTempView('desligados_1')

In [0]:
df_fecham_financ = spark.sql(f'''SELECT * FROM databox.juridico_comum.tb_fecham_financ_trab_{nmmes}''')

from pyspark.sql.functions import regexp_replace

# Substituir espaços da coluna "nome_coluna"
df_fecham_financ = df_fecham_financ.withColumn("MATRICULA", regexp_replace("MATRICULA", " ", ""))

df_fecham_financ.createOrReplaceTempView('df_fecham_financ')

In [0]:
%sql
SELECT * FROM df_fecham_financ LIMIT 10

ID_PROCESSO,DATACADASTRO,MES_CADASTRO,AREA_DO_DIREITO,SUB_AREA_DO_DIREITO,VALOR_DA_CAUSA,EMPRESA,STATUS,ADVOGADO_RESPONSAVEL,ESCRITORIO_RESPONSAVEL,NUMERO_DO_PROCESSO,PARTE_CONTRARIA_CPF,PARTE_CONTRARIA_NOME,ADVOGADO_PARTE_CONTRARIA,MATRICULA,BU,VP,DIRETORIA,DESCRICAO_CENTRO_CUSTO,RESPONSAVEL_AREA,AREA_FUNCIONAL,ESTADO,COMARCA,CLASSIFICACAO,NATUREZA_OPERACIONAL,TERCEIRO_PRINCIPAL,NOVO_TERCEIRO,TERCEIRO_AJUSTADO,DATA_ADMISSAO,DATA_DISPENSA,DATA_REGISTRADO,DISTRIBUICAO,PERIODO_RECLAMADO,PERIODO1,PERIODO2,PERIODO_RECL_GRUPO,PARTE_CONTRARIA_CARGO_GRUPO,CARGO,MOTIVO_DESLIGAMENTO,FASE,FASE_ATUAL,FILIAL,INDICACAO_PROCESSO_ESTRATEGICO,BANDEIRA,NOME_DA_LOJA,NOVOS,ENCERRADOS,ESTOQUE,DT_ULT_PGTO,DIRETORIA_DP_LOJA,MES_FECH,NOVO_X_LEGADO,PROCESSO_ESTEIRA,MES_DATA_ADMISSAO,MES_DATA_DISPENSA,TEMPO_EMPRESA_MESES,TEMPO_AJUIZAR_ACAO,PERC_SOCIO_M,PERC_EMPRESA_M,PROVISAO_MOV_M,PROVISAO_TOTAL_M_1,PROVISAO_TOTAL_M,CORRECAO_M_1,CORRECAO_M,CORRECAO_MOV_M,PROVISAO_TOTAL_PASSIVO_M,SOCIO_PROVISAO_TOTAL_M,EMPRESA_PROV_TOTAL_PASSIVO_M,EMPRESA_PROVISAO_MOV_M,ACORDO,CONDENACAO,PENHORA,OUTROS_PAGAMENTOS,IMPOSTO,GARANTIA,TOTAL_PAGAMENTOS,MESES_AGING_ENCERR,ANO_AGING_ENCERR,MESES_AGING_ESTOQ,ANO_AGING_ESTOQ,AGING_ESTOQ_MESES,MOTIVO_ENCERRAMENTO,FX_ANO_AGING_ENCERR,FX_MES_AGING_ESTOQ,FX_ANO_AGING_ESTOQ,FX_ANO_AGING_ESTOQ_2,FX_TEMPO_EMPRESA,DP_FASE,DP_VP_DIRETORIA,DP_NATUREZA,TRIM_ESTOQ,MOTIVO_ENC_AGRP,CARGO_TRATADO,CLUSTER_AGING,CLUSTER_VALOR,SAFRA_RECLAMACAO,ET
35068.0,2011-05-20,2011-05-01,TRABALHISTA,CONTENCIOSO INDIVIDUAL,40000.0,NOVA CASA BAHIA S/A (ATUAL VIA VAREJO),ATIVO,FERNANDA VALENTI TAVARES DE LIMA,FARIA ADVOGADOS E CONSULTORES DE EMPRESAS,0000435-62.2011.5.04.0030,586.275.450-49,MARLEI DE SA DA COSTA,ANA CRISTINA CONSUL NUNES,N/I,null,null,null,null,null,null,RS,PORTO ALEGRE,PRÓPRIO,VENDEDOR,null,null,null,2004-08-23,2009-08-28,2011-05-20,2011-05-04,2006-05-30,2006-05-30,2009-08-28,05-2006 a 08-2009,VENDEDOR,null,DISPENSA SEM JUSTA CAUSA,RECURSAL TST,EXECUÇÃO DEFINITIVA,1362,null,CB,CB CRICIUMA,null,null,1.0,null,DIRETORIA 5,null,LEGADO,MASSA - GERAL - FK,2004-08-01,2009-08-01,60,20,1.0,0.0,-28641.88,127296.067472,88229.28,10424.907472,0.0,0.0,98654.187472,23177.36856318519,0.00,-21117.80852279286,0.00,0.0,0.00,0.00,0.00,0.0,0.0,null,null,null,null,null,,null,null,null,null,5 - 6 anos,RECURSAL_TST,null,PASSÍVEL PROVISÃO,null,null,VENDEDOR,Sem info,MEDIO VALOR,2009,Sem info
223290.0,2015-02-23,2015-02-01,TRABALHISTA COLETIVO,PROCESSO JUDICIAL,32000.0,NOVA CASA BAHIA S/A (ATUAL VIA VAREJO),ATIVO,DENIZE WILL FURUKAWA,"PIPEK, PENTEADO E PAES MANSO ADVOGADOS",1000148-73.2015.5.02.0232,N/I,SINDICATO DOS EMPREGADOS NO COMERCIO DE OSASCO E REGIAO - 01/09/2009 - 31/08/2015,null,N/I,null,null,null,null,null,null,SP,CARAPICUÍBA,SINDICATO,SINDICATO / MINISTERIO PUBLICO,null,null,null,2009-09-01,2015-08-31,2015-02-23,2015-02-03,2010-03-01,2010-03-01,2015-08-31,03-2010 a 08-2015,null,null,ATIVO,RECURSAL TST,EXECUÇÃO DEFINITIVA (TST),1456,null,CB,JURIDICO LOJA GDE SP,null,null,1.0,null,DIRETORIA 1,null,LEGADO,COLETIVO,2009-09-01,2015-08-01,71,-6,0.167637126541236,0.832362873458764,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.0,null,null,null,null,null,,null,null,null,null,6 - 10 anos,RECURSAL_TST,null,ESTRATEGICO,null,null,OUTROS,Sem info,SP,2010,Sem info
223292.0,2015-02-23,2015-02-01,TRABALHISTA COLETIVO,PROCESSO JUDICIAL,32000.0,NOVA CASA BAHIA S/A (ATUAL VIA VAREJO),ATIVO,DENIZE WILL FURUKAWA,"PIPEK, PENTEADO E PAES MANSO ADVOGADOS",'10001478820155020232,N/I,SINDICATO DOS EMPREGADOS NO COMERCIO DE OSASCO E REGIAO - 01/09/2009 - 31/08/2015,null,N/I,null,null,null,null,null,null,SP,CARAPICUÍBA,SINDICATO,SINDICATO / MINISTERIO PUBLICO,null,null,null,2009-09-01,2015-08-31,2015-02-23,2015-02-03,2010-03-01,2010-03-01,2015-08-31,03-2010 a 08-2015,null,null,ATIVO,EXECUÇÃO PROVISÓRIA,EXECUÇÃO DEFINITIVA,1456,null,CB,JURIDICO LOJA GDE SP,null,null,1.0,null,DIRETORIA 1,null,LEGADO,COLETIVO,2009-09-01,2015-08-01,71,-6,0.167637126541236,0.8323628

In [0]:
df_fecham_financ_cons_1 = spark.sql('''
    select A.ID_PROCESSO, coalesce(A.MATRICULA, B.MATRICULA) AS MATRICULA, cast(coalesce(A.MATRICULA,B.MATRICULA) as int) as MATRICULAINT, coalesce(A.PARTE_CONTRARIA_CPF, B.PARTE_CONTRARIA_CPF) AS CPF, A.MES_FECH, A.NOVOS from df_fecham_financ A
    LEFT JOIN databox.juridico_comum.trab_ger_consolida_20250623 B
    ON A.ID_PROCESSO = B.PROCESSO_ID
''') 

from pyspark.sql.functions import regexp_replace

# Substituir espaços da coluna "nome_coluna"
df_fecham_financ_cons_1 = df_fecham_financ_cons_1.withColumn("CPF", regexp_replace("CPF",  r"[.,-]", ""))

df_fecham_financ_cons_1.createOrReplaceTempView('df_fecham_financ_cons_1')

In [0]:
df_fecham_financ_cons_2 = spark.sql('''
    SELECT *
    ,CASE WHEN LENGTH(CPF) == 11 THEN CONCAT(SUBSTRING(CPF,1,3),'.',SUBSTRING(CPF,4,3),'.',SUBSTRING(CPF,7,3),'-',SUBSTRING(CPF,10,2))
        ELSE CPF END AS CPFN

    FROM df_fecham_financ_cons_1
''')

from pyspark.sql.functions import regexp_replace

# Remover todos os espaços (um ou mais) da coluna "nome"
df_fecham_financ_cons_2 = df_fecham_financ_cons_2.withColumn("MATRICULA", regexp_replace("MATRICULA", r"\s+", ""))

df_fecham_financ_cons_2.createOrReplaceTempView('df_fecham_financ_cons_2')

In [0]:
%sql
SELECT 
  CPF,    -- Esta é a coluna APÓS a limpeza de pontos/traços (da Célula 17)
  CPFN    -- Esta é a coluna APÓS a re-formatação (da Célula 18)
FROM df_fecham_financ_cons_2 
LIMIT 20

CPF,CPFN
22387229800,223.872.298-00
27168003836,271.680.038-36
34269931809,342.699.318-09
12130732860,121.307.328-60
20000679801,200.006.798-01
25174434833,251.744.348-33
28345172822,283.451.728-22
17517553801,175.175.538-01
25177626865,251.776.268-65
08923301710,089.233.017-10


In [0]:
desligados_2 = spark.sql('''
    select * 
    ,CASE WHEN LENGTH(CPFN) == 11 THEN CONCAT(SUBSTRING(CPFN,1,3),'.',SUBSTRING(CPFN,4,3),'.',SUBSTRING(CPFN,7,3),'-',SUBSTRING(CPFN,10,2))
            ELSE CPFN END AS CPFN2
    from desligados_1
''')

desligados_2 = desligados_2.withColumn("MATRICULA", lpad("MATRICULA", 8, "0"))

desligados_2.createOrReplaceTempView('desligados_2')

In [0]:
%sql
SELECT 
  CPF,    -- Este é o CPF "sujo" original da base de RH
  CPFN,   -- Este é o resultado da Célula 15
  CPFN2   -- Este é o resultado final da Célula 19 (que vai no JOIN)
FROM desligados_2
LIMIT 20

CPF,CPFN,CPFN2
8.21058711E10,82105871110,821.058.711-10
6.3975815949E10,63975815949,639.758.159-49
63975815949,63975815949,639.758.159-49
12607570727,12607570727,126.075.707-27
8.833279707E9,88332797079,883.327.970-79
93675470644,93675470644,936.754.706-44
8531414792,08531414792,085.314.147-92
07112011884,07112011884,071.120.118-84
1.3893219838E10,13893219838,138.932.198-38
21803341840,21803341840,218.033.418-40


In [0]:
# %sql

# select A.MATRICULA, B.MATRICULA, A.CPFN2, B.CPFN, B.ID_PROCESSO, B.MES_FECH
# FROM desligados_2 A
# LEFT JOIN df_fecham_financ_cons_2 B 
# ON A.CPFN2 = B.CPFN

In [0]:
df_desligados_com_processos = spark.sql('''
    SELECT 
    A.COD_EMPRESA
    ,A.MATRICULA
    ,A.NOME
    ,A.CPFN AS CPF
    ,A.ESCOLARIDADE
    ,A.GENERO
    ,A.DATA_NASCIMENTO as DATA_NASCIMENTO
    ,A.STATUS_COLABORADOR as STATUS
    ,A.DATA_ADMISSAO as DATA_ADMISSAO
    ,A.TEMPO_EMPRESA as TEMPO_EMPRESA
    ,A.DESC_FUNCAO_ATUAL as CARGO
    ,A.DATA_DESLIGAMENTO as DATA_DESLIGAMENTO
    ,A.TIPO_DESLIGAMENTO as TIPO_DESLIGAMENTO
    ,A.MOTIVO_DESLIGAMENTO
    ,A.AGRUP_FUNCAO as AGRUPAMENTO_CARGO
    ,A.MATRICULA_GESTOR as MATRICULA_GESTOR
    ,A.NOME_GESTOR as NOME_GESTOR
    ,A.COD_CENTRO_RESULTADO as COD_CENTRO_CUSTO
    ,A.ESTRUTURA as ESTRUTURA
    ,A.NIVEL1 AS DESCRICAON1
    ,A.NIVEL2 AS DESCRICAON2
    ,A.NIVEL3 AS DESCRICAON3
    ,A.NIVEL4 AS DESCRICAON4
    ,A.COD_ESTABELECIMENTO AS ESTABELECIMENTO
    ,B.ID_PROCESSO
    ,B.MES_FECH
    ,coalesce(B.NOVOS, 0) AS NOVOS
    FROM desligados_2 A
    LEFT JOIN df_fecham_financ_cons_2 B
    ON A.CPFN2 = B.CPFN
''')

df_desligados_com_processos.createOrReplaceTempView('df_desligados_com_processos')

In [0]:
df_desligados_com_processos_f = spark.sql('''
    SELECT * 
        ,TRUNC(DATA_DESLIGAMENTO, 'MM') AS MES_DESLIGAMENTO
        ,ROW_NUMBER() OVER (PARTITION BY MATRICULA ORDER BY MES_FECH) AS RN
    FROM df_desligados_com_processos
''')

df_desligados_com_processos_f.createOrReplaceTempView('df_desligados_com_processos_f')



In [0]:
df_desligados_com_processos_ff = spark.sql('''
    SELECT *,
    CASE WHEN RN = 1 THEN 1
    ELSE 0 END AS QTDE

    FROM df_desligados_com_processos_f
''')


df_desligados_com_processos_ff.createOrReplaceTempView('df_desligados_com_processos_ff')

In [0]:
df_desligados_com_processos_ff.head()

Row(COD_EMPRESA=29.0, MATRICULA='001937.0', NOME='THIAGO RODRIGUES FERREIRA', CPF='12607570727', ESCOLARIDADE='COLEGIAL COMPLETO', GENERO='M', DATA_NASCIMENTO='1983-04-26 00:00:00', STATUS='AFASTADO', DATA_ADMISSAO='2015-07-17 00:00:00', TEMPO_EMPRESA='9.794520547945206', CARGO='AJUDANTE LOGISTICA EXT', DATA_DESLIGAMENTO='1900-01-01 00:00:00', TIPO_DESLIGAMENTO='', MOTIVO_DESLIGAMENTO='', AGRUPAMENTO_CARGO='09-OPERACIONAL', MATRICULA_GESTOR='60009190', NOME_GESTOR='MARCELO RODRIGUES DA CONCEICAO QUINTELA', COD_CENTRO_CUSTO=None, ESTRUTURA='LOGISTICA', DESCRICAON1='LOGISTICA', DESCRICAON2='DIR EXEC LOGISTICA E SUPPLY', DESCRICAON3='DISTRIBUICAO', DESCRICAON4='DIR GER TRANSP CTG', ESTABELECIMENTO='1401.0', ID_PROCESSO=632641.0, MES_FECH=None, NOVOS=0.0, MES_DESLIGAMENTO=datetime.date(1900, 1, 1), RN=1, QTDE=1)

In [0]:
%sql

SELECT STATUS, sum(novos) as TOTAL FROM df_desligados_com_processos_f
group by all

STATUS,TOTAL
AFASTADO,1703.0
DESLIGADO,12191.0
ATIVO,2629.0


In [0]:
import pandas as pd
from shutil import copyfile
import xlsxwriter
# Convert PySpark DataFrame to Pandas DataFrame
pandas_df = df_desligados_com_processos_ff.toPandas()

# Save the Pandas DataFrame to an Excel file
local_path = f'/local_disk0/tmp/TB_DESLIGADOS.xlsx'
pandas_df.to_excel(local_path, index=False, sheet_name='DESLIGADOS_RH_F', engine='xlsxwriter')

# Copy the file from the local disk to the desired volume
volume_path = f'/Volumes/databox/juridico_comum/arquivos/trabalhista/bases_indicadores_resultado/TB_DESLIGADOS.xlsx'

copyfile(local_path, volume_path)

'/Volumes/databox/juridico_comum/arquivos/trabalhista/bases_indicadores_resultado/TB_DESLIGADOS.xlsx'